# Monitoreo y Mantenimiento de Arquitectura de Datos (Fundamentos)

## Objetivos
- Comprender por qué monitorear sistemas de datos (prevención vs reacción).
- Conocer conceptos básicos: logs, métricas, alertas, dashboards.
- Identificar qué es deuda técnica en datos y su impacto.
- Reconocer señales de sistemas que necesitan mantenimiento.
- Introducir mantenimiento preventivo vs correctivo.
- Ejemplos de evolución de sistemas de datos.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Cultura Organizacional Data-Driven (Fundamentos)](13_cultura_organizacional_data_driven.ipynb) | Siguiente: [Fundamentos de Arquitectura de Datos para IA/ML](15_fundamentos_arquitectura_datos_ia_ml.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. Objetivos
2. 1. ¿Por Qué Monitorear Sistemas de Datos?
3. 2. Conceptos Básicos de Monitoreo
4. 3. ¿Qué es Deuda Técnica en Datos?
5. 4. Señales de Sistemas que Necesitan Mantenimiento
6. 5. Mantenimiento Preventivo vs Correctivo
7. 6. Código: Simulador de Incidentes
8. 7. Evolución de Sistemas de Datos

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Monitoreo y Mantenimiento de Arquitectura de Datos (Fundamentos)"] --> C["Notebook siguiente"]
```


## 1. ¿Por Qué Monitorear Sistemas de Datos?

### Analogía: Auto sin Mantenimiento
- **Sin monitoreo:** No revisas aceite, frenos, neumáticos → falla inesperada en autopista.
- **Con monitoreo:** Dashboard alerta "Aceite bajo" → cambias aceite antes de daño al motor.

### Realidad en Datos
| Sin Monitoreo | Con Monitoreo |
|--------------|---------------|
| Pipeline falla en silencio, dashboards muestran datos viejos | Alerta en 5 min: "Pipeline X no actualizó desde 08:00" |
| Usuarios reportan: "Los números están mal" | Monitoreo de calidad detecta 20% nulos automáticamente |
| Disco lleno → sistema cae | Alerta a 80% de uso → limpias archivos temporales |
| No sabes cuánto cuesta el sistema | Dashboard de costos → optimizas queries caros |

### Beneficios
1. **Prevención:** Detectar problemas antes de que impacten usuarios.
2. **Tiempo de resolución:** MTTD (Mean Time To Detect) + MTTR (Mean Time To Resolve) reducidos.
3. **Confianza:** Usuarios confían si saben que el sistema está monitoreado.
4. **Optimización:** Identificar cuellos de botella, costos innecesarios.

## 2. Conceptos Básicos de Monitoreo

### 2.1 Logs (Registros)
**Definición:** Eventos textuales con timestamp.

**Ejemplo:**
```
2025-11-07 08:15:23 INFO  Pipeline sales_daily started
2025-11-07 08:17:45 ERROR Failed to connect to source DB: timeout
2025-11-07 08:18:02 INFO  Retrying connection (attempt 2/3)
2025-11-07 08:18:15 INFO  Connection successful
2025-11-07 08:25:10 INFO  Pipeline sales_daily completed: 1.2M rows processed
```

**Niveles de log:**
- **DEBUG:** Información detallada para desarrollo.
- **INFO:** Eventos normales (inicio, fin de proceso).
- **WARNING:** Situación anómala pero no crítica.
- **ERROR:** Fallo que impide operación.
- **CRITICAL:** Fallo grave que requiere intervención inmediata.

---

### 2.2 Métricas (Números)
**Definición:** Valores cuantitativos en el tiempo.

**Ejemplos:**
- **Throughput:** Filas procesadas/segundo.
- **Latencia:** Tiempo desde evento hasta disponibilidad en Gold.
- **Error rate:** % de queries que fallan.
- **Data quality:** % de completitud, precisión.
- **Costos:** $ por día de compute/storage.

**Visualización:**
```
Throughput (filas/min)
  5000 ┤         ╭───╮
  4000 ┤     ╭───╯   ╰──╮
  3000 ┤ ╭───╯          ╰─
  2000 ┤─╯
       └────────────────────
        08:00 09:00 10:00 11:00
```

---

### 2.3 Alertas
**Definición:** Notificaciones cuando métrica cruza umbral.

**Ejemplo:**
- **Condición:** Latencia promedio > 5 minutos por 10 minutos consecutivos.
- **Acción:** Enviar email a equipo de datos + mensaje Slack #alerts.

**Tipos:**
- **Umbral fijo:** CPU > 80%.
- **Umbral dinámico:** Throughput cae 30% vs promedio 7 días.
- **Anomalía:** Detección automática de comportamiento inusual.

---

### 2.4 Dashboards
**Definición:** Visualización consolidada de métricas clave.

**Dashboard Típico de Pipeline:**
```
╔═══════════════════════════════════════════════════════╗
║            Pipeline: sales_daily                      ║
╠═══════════════════════════════════════════════════════╣
║ Estado:        ✓ Running                             ║
║ Última ejecución: 08:25 (hace 15 min)                ║
║ Duración:      12 min (normal: 10-15 min)            ║
║ Filas procesadas: 1.2M                                ║
║ Calidad:       98.5% ✓ (objetivo: >98%)              ║
║ Errores:       0                                      ║
║                                                       ║
║ Throughput (últimas 24h):  ▁▂▃▅▇▆▅▃▂▁                ║
║ Latencia (P95):            3.2 min ✓                  ║
╚═══════════════════════════════════════════════════════╝
```

## 3. ¿Qué es Deuda Técnica en Datos?

### Definición
**Deuda técnica:** Atajos o decisiones subóptimas que aceleran entrega inicial pero generan costos de mantenimiento futuros.

### Ejemplos en Datos
| Decisión Rápida ("Deuda") | Consecuencia Futura |
|---------------------------|---------------------|
| Hardcodear valores en código en lugar de configuración | Cambiar parámetro requiere redesplegar todo |
| No documentar transformaciones | Nadie entiende qué hace el pipeline 6 meses después |
| Copiar-pegar código entre pipelines | Bug se replica en 10 pipelines |
| Saltarse validación de calidad "por urgencia" | Datos erróneos en producción, pérdida de confianza |
| No versionar schemas | Cambio rompe 5 dashboards sin previo aviso |
| Ignorar alertas "ruidosas" | Alerta crítica real pasa desapercibida |

### Impacto
- **Tiempo de desarrollo:** Nuevas features toman 3x más tiempo.
- **Incidentes:** Más frecuentes y difíciles de debuggear.
- **Costos:** Sistemas ineficientes consumen más recursos.
- **Moral:** Equipo frustrante trabajando en código legacy frágil.

## 4. Señales de Sistemas que Necesitan Mantenimiento

### Síntomas de Deterioro
1. **Incidentes frecuentes:** >1 por semana en mismo pipeline.
2. **Tiempo de desarrollo creciente:** Feature simple toma semanas.
3. **Documentación desactualizada:** "Esto no coincide con el código".
4. **Miedo a cambiar:** "Si toco esto, puede romperse todo".
5. **Dependencias obsoletas:** Librerías sin actualizar por años.
6. **Performance degradado:** Queries que tardaban 5s ahora tardan 30s.
7. **Costos crecientes sin explicación:** Factura de cloud +50% sin nuevos proyectos.

### Cuestionario de Salud
Responde sí/no:
- [ ] ¿Puedes desplegar cambios con confianza?
- [ ] ¿Tienes alertas automáticas para problemas críticos?
- [ ] ¿La documentación está actualizada?
- [ ] ¿Nuevos integrantes pueden entender el sistema en <1 semana?
- [ ] ¿Menos de 10% del tiempo se gasta apagando fuegos?

**Si ≥3 respuestas "No":** Sistema necesita mantenimiento urgente.

## 5. Mantenimiento Preventivo vs Correctivo

### Mantenimiento Correctivo (Reactivo)
**Cuándo:** Después de que algo falla.

**Ejemplo:**
- Pipeline falla → equipo debuggea 3 horas → hotfix.
- Costo: 3h ingeniería + impacto en usuarios.

### Mantenimiento Preventivo (Proactivo)
**Cuándo:** Antes de que falle.

**Actividades:**
1. **Monitoreo continuo:** Detectar anomalías tempranas.
2. **Limpieza regular:** Borrar datos temporales, logs antiguos.
3. **Refactoring:** Simplificar código complejo antes de que cause problemas.
4. **Actualización de dependencias:** Aplicar parches de seguridad.
5. **Pruebas de carga:** Simular picos de tráfico.
6. **Revisión de costos:** Identificar recursos subutilizados.

### Comparación
| Aspecto | Correctivo | Preventivo |
|---------|-----------|------------|
| **Costo** | Alto (urgencias) | Medio (planificado) |
| **Tiempo** | Impredecible | Predecible (20% sprint) |
| **Impacto** | Usuarios afectados | Sin impacto visible |
| **Ejemplo** | Reparar pipeline roto | Optimizar query lento |

**Regla 80/20:** 80% de incidentes futuros se previenen con 20% de tiempo en mantenimiento preventivo.

## 6. Código: Simulador de Incidentes

In [ ]:
import random
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import List

@dataclass
class Incident:
    timestamp: datetime
    severity: str  # Low, Medium, High, Critical
    component: str
    description: str
    mttr_minutes: int  # Mean Time To Resolve
    
def simulate_incidents(months: int, maintenance_level: str) -> List[Incident]:
    """Simula incidentes según nivel de mantenimiento."""
    incidents = []
    
    # Tasa de incidentes según mantenimiento
    if maintenance_level == "none":
        incident_rate = 8  # 8 incidentes/mes
        critical_prob = 0.3
    elif maintenance_level == "reactive":
        incident_rate = 5
        critical_prob = 0.2
    elif maintenance_level == "proactive":
        incident_rate = 2
        critical_prob = 0.05
    else:
        incident_rate = 1
        critical_prob = 0.01
    
    for month in range(months):
        num_incidents = random.randint(max(1, incident_rate-2), incident_rate+2)
        
        for _ in range(num_incidents):
            # Determinar severidad
            rand = random.random()
            if rand < critical_prob:
                severity = "Critical"
                mttr = random.randint(120, 480)  # 2-8 horas
            elif rand < critical_prob + 0.2:
                severity = "High"
                mttr = random.randint(60, 180)
            elif rand < critical_prob + 0.5:
                severity = "Medium"
                mttr = random.randint(30, 90)
            else:
                severity = "Low"
                mttr = random.randint(10, 30)
            
            timestamp = datetime.now() - timedelta(days=30*(months-month)) + timedelta(days=random.randint(0, 29))
            
            incidents.append(Incident(
                timestamp=timestamp,
                severity=severity,
                component=random.choice(["Pipeline", "Database", "API", "Storage"]),
                description=f"Issue in {random.choice(['ingestion', 'transformation', 'serving'])}",
                mttr_minutes=mttr
            ))
    
    return sorted(incidents, key=lambda x: x.timestamp)

def analyze_incidents(incidents: List[Incident]) -> dict:
    total = len(incidents)
    critical = sum(1 for i in incidents if i.severity == "Critical")
    total_downtime = sum(i.mttr_minutes for i in incidents)
    
    return {
        "total_incidents": total,
        "critical_incidents": critical,
        "avg_mttr_hours": round(total_downtime / total / 60, 2) if total > 0 else 0,
        "total_downtime_hours": round(total_downtime / 60, 2)
    }

# Simulación: 6 meses
print("=== Impacto del Mantenimiento en Incidentes (6 meses) ===")
print()

for level, desc in [("none", "Sin mantenimiento"), 
                     ("reactive", "Correctivo"), 
                     ("proactive", "Preventivo")]:
    incidents = simulate_incidents(6, level)
    stats = analyze_incidents(incidents)
    
    print(f"{desc}:")
    print(f"  Incidentes totales:  {stats['total_incidents']}")
    print(f"  Incidentes críticos: {stats['critical_incidents']}")
    print(f"  MTTR promedio:       {stats['avg_mttr_hours']}h")
    print(f"  Downtime acumulado:  {stats['total_downtime_hours']}h")
    print()

print("Conclusión: Mantenimiento preventivo reduce incidentes 75% y downtime 80%.")
print("Costo: ~20% tiempo equipo en mantenimiento vs 40%+ apagando fuegos.")

## 7. Evolución de Sistemas de Datos

### Ciclo de Vida Típico
```
┌─────────────┐
│  Diseño     │  Semanas 1-4: Arquitectura inicial
└──────┬──────┘
       │
┌──────▼──────┐
│ Implementación │ Meses 1-3: Pipelines, dashboards
└──────┬──────┘
       │
┌──────▼──────┐
│  Operación  │  Meses 4-12: Uso productivo
└──────┬──────┘
       │
┌──────▼──────┐
│ Mantenimiento │ Continuo: Incidentes, optimización
└──────┬──────┘
       │
┌──────▼──────┐
│  Evolución  │  Año 2+: Nuevos features, refactoring
└──────┬──────┘
       │
┌──────▼──────┐
│ Reemplazo?  │  Año 5+: Migración a nueva arquitectura
└─────────────┘
```

### Drivers de Evolución
1. **Crecimiento de datos:** 10x volumen → necesita particionamiento.
2. **Nuevos casos de uso:** Agregar streaming cuando todo era batch.
3. **Tecnología obsoleta:** Hadoop → Spark → Cloud lakehouse.
4. **Requisitos regulatorios:** GDPR requiere lineage, derecho al olvido.
5. **Costos:** Migrar a tecnología más eficiente.

### Ejemplo: Evolución E-commerce
| Año | Arquitectura | Razón del Cambio |
|-----|--------------|------------------|
| **2015** | MySQL + cron jobs → CSV reports | Inicio, 1K pedidos/día |
| **2017** | Redshift + Airflow | Crecimiento: 10K pedidos/día, necesita BI |
| **2019** | Redshift + S3 data lake | Datos no estructurados (clicks, logs) |
| **2021** | Lakehouse (Delta + Spark) | Unificar batch + streaming, reducir costos |
| **2023** | Lakehouse + ML platform | Personalización, forecast con ML |
| **2025** | Cloud-native + real-time | Decisiones sub-segundo (pricing dinámico) |

## 8. Primeros Pasos en Monitoreo

### Para Comenzar (Semana 1)
1. **Identifica pipelines críticos:** Los 3 más importantes para negocio.
2. **Configura alertas básicas:**
   - Pipeline no ejecutó en horario esperado.
   - Calidad de datos <95%.
   - Costos diarios >X% del presupuesto.
3. **Dashboard simple:** Estado de pipelines (verde/rojo).
4. **Log centralized:** Todos los logs en un lugar (no dispersos).

### Para Equipos (Mes 1)
1. **Postmortem de incidentes:** Documentar qué falló, por qué, cómo prevenir.
2. **Reservar 20% tiempo:** Sprint planning incluye "tech debt" tasks.
3. **Actualizar documentación:** README de cada pipeline con owner, propósito, dependencias.
4. **Revisión mensual:** Métricas de salud (incidentes, downtime, costos).

## 9. Ejercicio Práctico
**Contexto:** Eres responsable de un pipeline de ventas que carga datos cada noche a las 02:00.

**Tareas:**
1. Lista 5 métricas que monitorearías.
2. Define 3 alertas con umbrales específicos.
3. Identifica 2 ejemplos de deuda técnica que podrías tener.
4. Propón 3 actividades de mantenimiento preventivo (mensual).
5. Diseña dashboard simple (en texto) con 5 paneles.

**Ejemplo de respuesta:**
- **Métrica 1:** Duración del pipeline (objetivo: 10-15 min).
- **Alerta 1:** Si duración >20 min, enviar Slack a #data-ops.
- **Deuda técnica 1:** Credenciales hardcodeadas en script (debería usar secrets manager).
- **Mantenimiento 1:** Revisar queries lentas y optimizar (última semana del mes).

## 10. Resumen
- **Monitoreo previene problemas:** Logs, métricas, alertas, dashboards.
- **Deuda técnica:** Atajos que generan costos futuros de mantenimiento.
- **Señales de deterioro:** Incidentes frecuentes, desarrollo lento, miedo a cambiar.
- **Preventivo > Correctivo:** 20% tiempo en mantenimiento previene 80% incidentes.
- **Evolución continua:** Sistemas de datos cambian cada 2-3 años por crecimiento/tecnología.

**Siguiente (Mid):** Observabilidad aplicada, stack tecnológico, gestión de deuda técnica, versionado de schemas.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Referencias del curso: ../04-recursos/referencias.md
